# Ejercicio: Asistente de Documentos con Gemini y Gradio
 
**Tiempo:** 40-45 minutos  
**Grupos:** 3-4 personas  
 
## Contexto
 
En los notebooks anteriores construimos chatbots que responden preguntas generales.
En este ejercicio daremos un paso más: construir un asistente que responda preguntas
sobre un documento específico — en este caso, el paper seminal de los Transformers:
**"Attention is All You Need"** (Vaswani et al., 2017).
 
Esta técnica — inyectar el contenido de un documento en el contexto del modelo —
es la base conceptual de **RAG (Retrieval-Augmented Generation)**, uno de los
patrones más usados en aplicaciones de IA en producción.
 
## Objetivo
 
Construir una aplicación Gradio donde el usuario pueda:
1. Cargar el PDF del paper
2. Hacer preguntas sobre su contenido
3. Obtener respuestas basadas **exclusivamente** en el documento
 
## Lo que van a aprender
 
- Extracción de texto desde PDFs con `pypdf`
- Inyección de contexto externo en el system prompt
- Limitaciones de este enfoque y por qué existe RAG
- Integración de `gr.File` en una interfaz Gradio
 
---

## Paso 0: Instalación y configuración
 
### 0.1 Instala las dependencias necesarias
 
Necesitarás tres bibliotecas nuevas además de las que ya conoces:
- `pypdf`: para extraer texto de archivos PDF
- `gradio`: para la interfaz web
- `google-genai`: para el modelo
 
```
pip install pypdf gradio google-genai python-dotenv
```

In [ ]:
# %%bash
# pip install pypdf gradio google-genai python-dotenv

### 0.2 Descarga el paper
 
Descarga el PDF de ArXiv (acceso abierto):
```
https://arxiv.org/pdf/1706.03762
```
 
Guárdalo en la misma carpeta que este notebook con el nombre `attention_is_all_you_need.pdf`.
 
### 0.3 Configura tus credenciales
 
Crea un archivo `.env` con tu API key de Gemini:
```
GEMINI_API_KEY="tu_key_aqui"
```

## Paso 1: Extracción de texto del PDF

Lo primero es leer el PDF y extraer su contenido como texto plano.
`pypdf` hace esto en pocas líneas.

**Instrucciones:**
1. Importa `PdfReader` desde `pypdf`
2. Crea una función `extract_text_from_pdf(pdf_path)` que:
   - Abra el PDF desde la ruta `pdf_path`
   - Itere sobre todas las páginas
   - Concatene el texto de cada página
   - Retorne el texto completo como string
3. Prueba la función con `attention_is_all_you_need.pdf`
4. Imprime los primeros 500 caracteres para verificar que funcionó

**Pista:** `PdfReader` tiene un atributo `pages` que es una lista.
Cada página tiene un método `extract_text()`.

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extrae todo el texto de un archivo PDF.
    
    Args:
        pdf_path: Ruta al archivo PDF
    
    Returns:
        Texto completo del PDF como string, todas las páginas concatenadas
    """
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        # extract_text() retorna None en páginas solo-imagen; "or ''" evita TypeError
        text += (page.extract_text() or "") + "\n"
    return text

# Prueba la función
pdf_path = "data/attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

# Estadísticas
reader = PdfReader(pdf_path)
print(f"Numero de paginas: {len(reader.pages)}")
print(f"Caracteres extraidos: {len(document_text):,}")

# Verificar primeros 500 caracteres
print("\n--- Primeros 500 caracteres ---")
print(document_text[:500])

## Paso 2: Inicialización del cliente de Gemini

Igual que en los notebooks anteriores.

**Instrucciones:**
1. Importa las bibliotecas necesarias (`os`, `dotenv`, `google.genai`, `google.genai.types`)
2. Carga las variables de entorno
3. Inicializa el cliente de Gemini
4. Define la constante `MODELO = "gemini-2.5-flash-lite"`
5. Verifica que la API key esté disponible

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que la API key esté disponible
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

# Inicializar el cliente de Gemini
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Modelo que usaremos en todo el notebook
MODELO = "gemini-2.5-flash-lite"

## Paso 3: Función de chat con contexto del documento

Esta es la parte central del ejercicio. La idea es construir un system prompt
que incluya el texto completo del paper, instruyendo al modelo a responder
**solo** basándose en ese contenido.

**Instrucciones:**
1. Crea una función `build_system_prompt(document_text)` que reciba el texto
   del documento y retorne un system prompt que:
   - Defina el rol del asistente (experto en el paper)
   - Incluya el texto completo del documento
   - Instruya al modelo a responder **solo** con información del documento
   - Indique qué hacer si la respuesta no está en el documento

2. Crea una función `chat_con_documento(message, history, document_text)` que:
   - Construya el historial en formato Gemini (objetos `types.Content`)
   - Use `generate_content_stream` con el system prompt del documento
   - Retorne la respuesta con `yield` para streaming

**Pista:** Recuerda que en Gradio el historial llega como lista de dicts
con claves `role` y `content`. El rol del asistente en Gemini es `"model"`.

In [ ]:
def get_text(content) -> str:
    """Extrae texto de un contenido de mensaje de Gradio,
    sin importar si llega como string o lista de dicts."""
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        return content[0].get("text", "") if content else ""
    return str(content)


def build_system_prompt(document_text: str) -> str:
    """Construye el system prompt que incluye el texto completo del documento."""
    return f"""Eres un experto asistente académico especializado en el paper
"Attention is All You Need" (Vaswani et al., 2017).

A continuación se te proporciona el texto COMPLETO del paper:

--- INICIO DEL DOCUMENTO ---
{document_text}
--- FIN DEL DOCUMENTO ---

INSTRUCCIONES IMPORTANTES:
1. Responde EXCLUSIVAMENTE con información que esté en el documento proporcionado arriba.
2. Si la pregunta del usuario no puede ser respondida con la información del documento,
   responde: "La información solicitada no se encuentra en el documento proporcionado."
3. NO uses tu conocimiento general para responder, incluso si conoces la respuesta.
4. Cuando sea relevante, menciona la sección o parte del paper donde se encuentra la información.
5. Responde en el mismo idioma en que el usuario hace la pregunta.
6. Incluye siempre una cita textual del documento (entre comillas) que respalde tu respuesta."""


def chat_con_documento(message: str, history: list, document_text: str):
    """Función de chat que Gradio llama cada vez que el usuario envía un mensaje.
    
    Args:
        message: Mensaje actual del usuario
        history: Historial en formato Gradio 5.x
        document_text: Texto del documento (pasado vía additional_inputs)
    
    Yields:
        Respuesta acumulada del modelo (streaming)
    """
    # 1. Construir system prompt con el documento completo incrustado
    system_prompt = build_system_prompt(document_text)

    # 2. Convertir historial de Gradio al formato Gemini (lista de types.Content)
    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            # MAPEO CRÍTICO: Gradio usa "assistant", Gemini requiere "model"
            # Si se envía "assistant", la API lanza error de rol inválido
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(
                    role=role,
                    parts=[types.Part(text=get_text(entry["content"]))]
                )
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            # Compatibilidad con formato alternativo [user_msg, assistant_msg]
            user_msg, assistant_msg = entry
            contenido.append(
                types.Content(role="user", parts=[types.Part(text=get_text(user_msg))])
            )
            if assistant_msg:
                contenido.append(
                    types.Content(role="model", parts=[types.Part(text=get_text(assistant_msg))])
                )

    # 3. Agregar mensaje actual del usuario al final del historial
    contenido.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    # 4. Llamar a Gemini en modo streaming (yield activa streaming en Gradio automáticamente)
    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,  # Directiva de comportamiento, más peso que el user prompt
        ),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta  # Gradio detecta yield → muestra tokens progresivamente

## Paso 4: Interfaz Gradio

Ahora construimos la interfaz. Usaremos `gr.ChatInterface` con `additional_inputs`
para pasar el texto del documento extraído.

**Instrucciones:**
1. Extrae el texto del PDF usando la función del Paso 1
2. Imprime cuántas páginas tiene y cuántos caracteres extraíste
3. Construye la interfaz con `gr.ChatInterface`:
   - `fn`: tu función `chat_con_documento`
   - `title`: un título descriptivo
   - `description`: explica qué puede hacer el asistente
   - `additional_inputs`: un `gr.Textbox` visible=False que contenga el texto
     del documento (así Gradio lo pasa automáticamente a la función)
   - `examples`: al menos 3 preguntas relevantes sobre el paper

**Pista:** Para pasar el texto del documento sin mostrarlo en la interfaz:
```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz con `demo.launch(server_name="0.0.0.0", server_port=8080)`

In [ ]:
import gradio as gr

# Extraer texto del PDF
pdf_path = "data/attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

# Mostrar estadísticas del documento
reader = PdfReader(pdf_path)
print(f"Documento cargado: {len(reader.pages)} paginas, {len(document_text):,} caracteres")

# Crear la interfaz de chat
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del Paper: Attention is All You Need",
    description="""Pregúntale cualquier cosa sobre el paper \"Attention is All You Need\"
    (Vaswani et al., 2017). El asistente responde EXCLUSIVAMENTE con información del documento.""",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    examples=[
        "¿Cuál es la arquitectura principal propuesta en el paper?",
        "¿Qué es el mecanismo de atención multi-cabeza?",
        "¿Cuántas capas tiene el encoder del modelo base?",
        "¿Quiénes son los autores del paper?",
        "¿Cuál es el resultado del modelo en WMT 2014 English-to-German?",
    ],
    flagging_mode="never"
)

# Lanzar la interfaz
demo.launch(
    server_name="0.0.0.0",
    server_port=8080,
    show_error=True
)

## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

## Paso 6 (Adicional): Mejora el sistema

Las cuatro mejoras opcionales fueron implementadas:

**A) Indicador de tokens** ✅ **(Implementada)**
Usa `client.models.count_tokens()` para obtener el conteo real de tokens del system prompt
desde la API de Gemini (no una estimación). Muestra cuántos tokens usa el paper y qué
porcentaje representan del límite de 1,000,000 tokens de Gemini 2.5 Flash.

**B) Subida dinámica de PDF** ✅ **(Implementada)**
Nueva interfaz en el puerto 8081 con `gr.File` en `additional_inputs`. Cuando el usuario
sube un PDF, `chat_con_documento_v2` extrae su texto y lo usa como documento base.
Si no se sube nada, usa el paper "Attention is All You Need" por defecto.

**C) Citas del documento** ✅ **(Implementada)**
Instrucción 6 del system prompt obliga al modelo a incluir una cita textual del paper
entre comillas en cada respuesta, haciendo las respuestas verificables.

**D) Multi-PDF con citación de fuente** ✅ **(Implementada)**
Nueva interfaz en el puerto 8082 que carga todos los PDFs de la carpeta `data/` y permite
subir PDFs adicionales. Cuando responde, el modelo indica de cuál archivo proviene cada
afirmación usando el formato `[nombre_archivo.pdf]`, permitiendo consultar múltiples
documentos simultáneamente.

In [ ]:
# ============================================================
# MEJORA A: Indicador de tokens (conteo real con la API de Gemini)
# ============================================================

system_prompt_completo = build_system_prompt(document_text)

# count_tokens llama a la API y retorna el conteo exacto (no estimación)
token_response = client.models.count_tokens(
    model=MODELO,
    contents=system_prompt_completo
)
tokens_reales = token_response.total_tokens
LIMITE = 1_000_000  # Gemini 2.5 Flash: 1M tokens de contexto

print(f"=== INDICADOR DE TOKENS — MEJORA A ===")
print(f"Tokens del system prompt (conteo Gemini): {tokens_reales:,}")
print(f"Límite del modelo ({MODELO}):              {LIMITE:,}")
print(f"Porcentaje usado:                          {tokens_reales / LIMITE * 100:.2f}%")
print(f"Tokens disponibles para conversación:      {LIMITE - tokens_reales:,}")
print()
print(f"Conclusión: el paper usa solo el {tokens_reales/LIMITE*100:.1f}% del límite.")
print(f"Quedan {LIMITE - tokens_reales:,} tokens libres para el historial de chat.")

In [ ]:
# ============================================================
# MEJORA B: Subida dinámica de PDF
# ============================================================
# Nueva función de chat que acepta un archivo PDF opcional.
# Si el usuario sube un PDF → usa ese documento.
# Si no sube nada       → usa el paper por defecto (Attention is All You Need).

def chat_con_documento_v2(message: str, history: list, pdf_file, document_text_default: str):
    """Versión mejorada: acepta subida dinámica de PDF vía gr.File."""
    # Usar el PDF subido si existe, si no el texto por defecto
    if pdf_file is not None:
        texto = extract_text_from_pdf(pdf_file)  # pdf_file es la ruta temporal del archivo subido
    else:
        texto = document_text_default

    system_prompt = build_system_prompt(texto)

    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))])
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            u, a = entry
            contenido.append(types.Content(role="user", parts=[types.Part(text=get_text(u))]))
            if a:
                contenido.append(types.Content(role="model", parts=[types.Part(text=get_text(a))]))

    contenido.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta


demo_v2 = gr.ChatInterface(
    fn=chat_con_documento_v2,
    title="Asistente de Documentos — Subida Dinámica de PDF",
    description=(
        "Sube cualquier PDF y haz preguntas sobre su contenido. "
        "Si no subes ningún archivo, el asistente responde sobre 'Attention is All You Need'."
    ),
    additional_inputs=[
        gr.File(label="Sube un PDF (opcional)", file_types=[".pdf"]),   # Mejora B
        gr.Textbox(value=document_text, visible=False),                  # texto por defecto
    ],
    examples=[
        "¿De qué trata este documento?",
        "Haz un resumen de las ideas principales",
        "¿Cuáles son las conclusiones del documento?",
    ],
    flagging_mode="never"
)

demo_v2.launch(server_name="0.0.0.0", server_port=8081, show_error=True)

In [ ]:
# ============================================================
# MEJORA D: Multi-PDF — responde de cualquier documento y cita la fuente
# ============================================================

def load_pdfs_from_folder(folder: str) -> dict:
    """Carga todos los PDFs de una carpeta. Retorna {filename: text}."""
    docs = {}
    if os.path.isdir(folder):
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith(".pdf"):
                fpath = os.path.join(folder, fname)
                try:
                    docs[fname] = extract_text_from_pdf(fpath)
                except Exception as e:
                    print(f"Error leyendo {fname}: {e}")
    return docs


def build_system_prompt_multi(docs: dict) -> str:
    """Construye system prompt para múltiples documentos con etiquetas de origen."""
    bloques = []
    for i, (nombre, texto) in enumerate(docs.items(), 1):
        bloques.append(
            f"=== DOCUMENTO {i}: {nombre} ===\n{texto}\n=== FIN DOCUMENTO {i}: {nombre} ==="
        )
    documentos_str = "\n\n".join(bloques)
    nombres = ", ".join(docs.keys())

    return f"""Eres un asistente experto en análisis de documentos académicos.
Tienes acceso a {len(docs)} documento(s): {nombres}

{documentos_str}

INSTRUCCIONES IMPORTANTES:
1. Responde EXCLUSIVAMENTE con información que esté en los documentos proporcionados.
2. SIEMPRE indica de cuál documento proviene cada afirmación con el formato [nombre_archivo.pdf].
   Ejemplo: "Según [attention_is_all_you_need.pdf], el modelo usa 6 capas de encoder."
3. Si la respuesta involucra varios documentos, cita cada uno con su etiqueta.
4. Si la información no está en ningún documento, responde exactamente:
   "La información no se encuentra en los documentos cargados."
5. NO uses conocimiento general; cíñete al contenido de los documentos.
6. Incluye citas textuales (entre comillas) junto a la etiqueta del documento fuente."""


def chat_multi_pdf(message: str, history: list, uploaded_files, use_defaults: bool):
    """Chat que responde de múltiples PDFs e indica la fuente de cada respuesta."""
    docs = {}

    # Documentos base precargados de la carpeta data/
    if use_defaults:
        docs.update(base_docs)

    # PDFs subidos dinámicamente por el usuario
    if uploaded_files:
        files = uploaded_files if isinstance(uploaded_files, list) else [uploaded_files]
        for f in files:
            nombre = os.path.basename(f)
            try:
                docs[nombre] = extract_text_from_pdf(f)
            except Exception as e:
                docs[nombre] = f"[Error al leer el archivo: {e}]"

    if not docs:
        yield "No hay documentos cargados. Sube al menos un PDF o activa los documentos base."
        return

    system_prompt = build_system_prompt_multi(docs)

    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))]))
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            u, a = entry
            contenido.append(types.Content(role="user", parts=[types.Part(text=get_text(u))]))
            if a:
                contenido.append(types.Content(role="model", parts=[types.Part(text=get_text(a))]))

    contenido.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta


# Pre-cargar todos los PDFs de la carpeta data/
base_docs = load_pdfs_from_folder("data")
print(f"Documentos base cargados: {len(base_docs)}")
for nombre, texto in base_docs.items():
    print(f"  - {nombre} ({len(texto):,} caracteres)")

demo_multi = gr.ChatInterface(
    fn=chat_multi_pdf,
    title="Asistente Multi-Documento — Cita la fuente automáticamente",
    description=(
        "Pregunta sobre cualquier documento cargado. "
        "El asistente indica de cuál PDF proviene cada respuesta usando [nombre.pdf].\n"
        f"Documentos base: {', '.join(base_docs.keys())}"
    ),
    additional_inputs=[
        gr.File(
            label="Agregar PDFs adicionales (opcional)",
            file_types=[".pdf"],
            file_count="multiple",
        ),
        gr.Checkbox(label="Incluir documentos base de la carpeta data/", value=True),
    ],
    examples=[
        "¿De qué trata cada documento?",
        "¿Cuál es la idea principal del paper sobre transformers?",
        "Compara los enfoques de los diferentes documentos",
    ],
    flagging_mode="never",
)

demo_multi.launch(server_name="0.0.0.0", server_port=8082, show_error=True)

## Mejora D: Multi-PDF con citación automática de fuente

Extiende el sistema para manejar **múltiples documentos simultáneamente**.
Cuando el usuario hace una pregunta, el asistente responde citando explícitamente
de cuál PDF proviene cada afirmación usando el formato `[nombre_archivo.pdf]`.

- Carga automática de todos los PDFs en la carpeta `data/`
- Subida dinámica de PDFs adicionales vía `gr.File(file_count="multiple")`
- Checkbox para activar/desactivar los documentos base
- Sistema prompt con bloques etiquetados por nombre de archivo

## Reflexión final

### 1. ¿Cuál es la limitación principal de este enfoque?

El enfoque actual inyecta **todo el texto del documento en el system prompt**. Esto funciona
mientras el documento quepa en la ventana de contexto del modelo. Para un paper de ~15 páginas
usamos ~13,000 tokens — solo el 1.3% del límite de 1,000,000 tokens de Gemini 2.5 Flash.

Sin embargo, con un documento de **1,000 páginas** (~866,000 tokens), estaríamos al 86.6% del
límite y pagaríamos esos tokens **en cada pregunta**. Con 2,000+ páginas superaríamos el límite
y el sistema fallaría completamente. Además, el costo de la API crece linealmente con el tamaño
del contexto enviado.

### 2. ¿Por qué existe RAG?

**RAG (Retrieval-Augmented Generation)** resuelve el problema de la limitación del contexto:

1. **Indexación previa**: El documento se divide en fragmentos pequeños ("chunks") y se convierten
   en vectores numéricos (*embeddings*) que capturan el significado semántico.
2. **Búsqueda por relevancia**: Cuando el usuario hace una pregunta, se buscan los chunks más
   similares semánticamente a esa pregunta usando distancia coseno entre vectores.
3. **Contexto selectivo**: Solo los chunks relevantes (~3-5 fragmentos) se inyectan en el prompt,
   no el documento completo.

Resultado: en vez de enviar 866,000 tokens por pregunta, se envían ~2,000 tokens con solo la
información pertinente. Escala a documentos de millones de páginas.

### 3. ¿Qué información podría "filtrarse" aunque el system prompt diga que no?

Gemini fue entrenado con texto de internet, que incluye el paper "Attention is All You Need"
(es un paper famoso y público desde 2017). Por tanto, el modelo **ya conoce** el contenido
del paper antes de leer el system prompt.

Si el system prompt dice "responde solo con el documento", el modelo tiende a obedecer, pero:
- Puede completar información que no está literalmente en el PDF (por ej., contexto histórico)
- No podemos distinguir si cita el documento o su conocimiento previo
- La instrucción 3 mitiga esto pero no garantiza aislamiento perfecto

**Para verificarlo:** Probar el mismo chatbot con un documento ficticio inventado (datos que el
modelo no puede conocer de su entrenamiento). Si responde correctamente a preguntas sobre ese
documento, está leyendo el contexto. Si "inventa" respuestas, está usando conocimiento previo.

---

## Recursos

- [Documentación de pypdf](https://pypdf.readthedocs.io)
- [Documentación de Gradio](https://www.gradio.app/docs)
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [Paper original en ArXiv](https://arxiv.org/abs/1706.03762)